# 04 Marker 空间表达

依次检查 Myl7、Tnnt2、Mef2c、Shh、Cer1、Apela，只绘制真实存在的基因。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
from create_sample_data import synthetic
from h5ad_utils import choose_h5ad
synthetic_path = ROOT / 'data' / 'processed' / 'synthetic_test.h5ad'
if not list((ROOT / 'data' / 'external' / 'GSE278603' / 'h5ad').glob('*.h5ad')):
    synthetic(synthetic_path)
H5AD = choose_h5ad(ROOT)
IS_SYNTHETIC = H5AD.name == 'synthetic_test.h5ad'
print('数据文件：', H5AD)
print('注意：' if IS_SYNTHETIC else '状态：', '当前使用完全合成测试数据' if IS_SYNTHETIC else '当前使用真实 GEO 数据')


In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei','SimHei','DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
from h5ad_utils import gene_vector, select_coordinates
markers = ['Myl7','Tnnt2','Mef2c','Shh','Cer1','Apela']
adata = ad.read_h5ad(H5AD, backed='r')
coords, source = select_coordinates(adata, 2)
available = [g for g in markers if g in adata.var_names]
if not available:
    raise KeyError('六个候选 marker 均不在 var_names；请检查基因标识类型')
fig, axes = plt.subplots(2, 3, figsize=(13,8), constrained_layout=True)
for ax, gene in zip(axes.flat, available):
    values = gene_vector(adata, gene)
    p = ax.scatter(coords[:,0], coords[:,1], c=values, s=4, cmap='magma')
    ax.set_title(gene); ax.set_aspect('equal'); fig.colorbar(p, ax=ax, shrink=.7)
for ax in axes.flat[len(available):]: ax.axis('off')
fig.suptitle(('合成测试：' if IS_SYNTHETIC else '') + 'Marker 空间表达')
out = ROOT / 'results' / 'figures' / 'notebook_marker_expression.png'
fig.savefig(out, dpi=180); plt.show()
adata.file.close()
out
